In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yt
from scipy.spatial.distance import cdist
from scipy.interpolate import CubicSpline
from astropy.constants import G
from tqdm import tqdm
import healpy as hp
import os
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu, bws_test
from scipy.stats import norm
from scipy.spatial import ConvexHull, Delaunay
from yt.data_objects.particle_filters import add_particle_filter
if int((yt.__version__).split('.')[0]) >= 4 and int((yt.__version__).split('.')[1]) >= 2: #ParticleUnion is only available in yt 4.2 and later
    from yt.data_objects.unions import ParticleUnion
else:
    from yt.data_objects.unions import Union as ParticleUnion

import setup
from importlib import reload
reload(setup)

from setup import load_halotree_and_pfs, load_timings, load_ds, load_tracking_dist_data, sec_branch_compute
from setup import gas_name_dict, gas_temp_dict
from setup import add_metallicity_fields, add_cooling_fields, add_radialdist_to_halocenter_field, extract_and_order_snapshotIdx
from setup import get_haloradius, infall_timestep_compute_hullv
from setup import codetp_list, label_list, color_list, marker_list

In [ ]:
def std_weighted(values, weights):
    average = np.average(values, weights=weights, axis=0)
    variance = np.average((values - average)**2, weights=weights, axis=0)*len(weights)/(len(weights) - 1)
    return np.sqrt(variance)

### The indices in parentheses are the infalling timesteps (t_begin) for non-spherical halos, found by using this code
GIZMO (117), GEAR (326), ENZO (123), RAMSES (119), GADGET3 (108), ART (123), AREPO (110), AREPO-TNG (112), CHANGA (180), GADGET4 (105)

In [ ]:
codetp = 'ENZO'
merger_number = '0'
halotree_ver = 2013

In [ ]:
rawtree, redshift_list, time_list, pfs, step = load_halotree_and_pfs(codetp, halotree_ver)
prog_branch, sec_branch, sec_branch_2 = sec_branch_compute(codetp, merger_number)
assignment = np.load('/work/hdd/bezm/gtg115x/Halo_Finding/%s/star_id_%s_final_firstmerger.npy' % (codetp, halotree_ver), allow_pickle=True).tolist()
hullv = np.load('/work/hdd/bezm/gtg115x/Halo_Finding/%s/hullv_%s_final.npy' % (codetp, halotree_ver), allow_pickle=True).tolist()

In [ ]:
#Use this to compute the infall timestep for spherical halos, as an estimate for the infall timestep for non-spherical halos.
from setup import infall_timestep_compute_spherical
infall_timestep_compute_spherical(rawtree, prog_branch, '0_61', step, halo_radius = True, printerror=True)

In [ ]:
idx_test = 123
ds = load_ds(codetp, idx_test, pfs)
meter = ds.length_unit.in_units('m')

def load_dm_particles(left, right):
    save_part = '/work/hdd/bezm/gtg115x/Halo_Finding/%s/particle_save/' % (codetp)
    if os.path.exists(save_part + 'part_dict.npy'):
        part_dict = np.load(save_part + 'part_dict.npy', allow_pickle=True).tolist()
        ind_array = np.arange(len(part_dict[idx_test]))
        ll = part_dict[idx_test]['ll']
        ur = part_dict[idx_test]['ur']
        ind_array = np.arange(len(ll))
        bool_overlap = (np.sum( ur <= left,axis=1)==0)*(np.sum(ll >= right,axis=1)==0)
        ind_array = ind_array[bool_overlap]
        mass,pos,vel,ids = np.array([]),np.array([[]]),np.array([[]]),np.array([])
        #
        for i in ind_array:
            part = np.load(save_part+'/part_%s_%s.npy' % (idx_test,i),allow_pickle= True).tolist()
            if len(mass) ==0:
                mass = part['mass']
                pos = part['pos']
                vel = part['vel']
                ids = part['ids']
            else:
                mass = np.append(mass,part['mass'])
                pos = np.vstack((pos,part['pos']))
                vel = np.vstack((vel,part['vel']))
                ids = np.append(ids,part['ids'])
        if len(mass)>0 and len(pos)>0:
            bool_in = (np.sum(pos >= left*meter,axis=1) ==3)*(np.sum(pos < right*meter,axis=1) ==3)
            mass,pos,vel,ids = mass[bool_in],pos[bool_in],vel[bool_in],ids[bool_in]
    else:
        reg = ds.box(left, right)
        dm_name_dict = {'ENZO':'DarkMatter','GEAR': 'DarkMatter', 'GADGET3': 'DarkMatter', 'AREPO': 'DarkMatter', 'AREPO-TNG': 'DarkMatter', 'GIZMO': 'DarkMatter', 'RAMSES': 'DM', 'ART': 'darkmatter', 'CHANGA': 'DarkMatter'}
        if codetp == 'ENZO':
            def darkmatter_init(pfilter, data):
                filter_darkmatter0 = np.logical_or(data["all", "particle_type"] == 1, data["all", "particle_type"] == 4)
                filter_darkmatter = np.logical_and(filter_darkmatter0,data['all', 'particle_mass'].to('Msun') > 1)
                return filter_darkmatter
            add_particle_filter("DarkMatter",function=darkmatter_init,filtered_type='all',requires=["particle_type","particle_mass"])
            ds.add_particle_filter("DarkMatter")
        if codetp == 'AREPO' or codetp == 'AREPO-TNG':
            dm = ParticleUnion("DarkMatter",["PartType2","PartType1"])
            ds.add_particle_union(dm)
        mass = reg[dm_name_dict[codetp], 'particle_mass'].to('kg').v
        pos = reg[dm_name_dict[codetp], 'particle_position'].to('m').v
        vel = reg[dm_name_dict[codetp], 'particle_velocity'].to('m/s').v
        ids = reg[dm_name_dict[codetp], 'particle_index'].v.astype(int) 
    return mass, pos, vel, ids

if codetp == 'ENZO' or codetp == 'RAMSES' or codetp == 'ART':
    mass, pos, vel, ids = load_dm_particles(rawtree[prog_branch][idx_test]['Halo_Center'] - rawtree[prog_branch][idx_test]['Halo_Radius']*3, rawtree[prog_branch][idx_test]['Halo_Center'] + rawtree[prog_branch][idx_test]['Halo_Radius']*3)
else:
    mass, pos, vel, ids = load_dm_particles(rawtree[prog_branch][idx_test]['Halo_Center'] - rawtree[prog_branch][idx_test]['Halo_Radius']*1e99, rawtree[prog_branch][idx_test]['Halo_Center'] + rawtree[prog_branch][idx_test]['Halo_Radius']*1e99)


#Extract convex hulls
hullv_pos_prog = pos[np.intersect1d(hullv[prog_branch][idx_test], ids, return_indices=True)[2]]
hullv_pos_sec = pos[np.intersect1d(hullv[sec_branch][idx_test], ids, return_indices=True)[2]]
print('prog intersect check:', len(hullv_pos_prog)/len(hullv[prog_branch][idx_test])) #should be 1.0 if all particles are found in the region
print('sec intersect check:', len(hullv_pos_sec)/len(hullv[sec_branch][idx_test])) #should be 1.0 if all particles are found in the region


def normalize(v):
    n = np.linalg.norm(v)
    if n < 1e-12:
        return None
    return v / n

def project_points(points, axis):
    """Project 3D points onto a 3D axis and return min/max projections."""
    projections = points @ axis
    return projections.min(), projections.max()

def intervals_overlap(a_min, a_max, b_min, b_max):
    return not (a_max < b_min or b_max < a_min)

def get_face_normals(hull):
    normals = []
    for simplex in hull.simplices:
        pts = hull.points[simplex]
        # Compute face normal from triangle
        v1 = pts[1] - pts[0]
        v2 = pts[2] - pts[0]
        n = np.cross(v1, v2)
        n = normalize(n)
        if n is not None:
            normals.append(n)
    return normals

def get_edges(hull):
    edges = set()
    for simplex in hull.simplices:
        i, j, k = simplex
        edges.add(tuple(sorted((i, j))))
        edges.add(tuple(sorted((j, k))))
        edges.add(tuple(sorted((k, i))))
    # Convert to vectors
    edge_vectors = []
    for i, j in edges:
        v = hull.points[j] - hull.points[i]
        if np.linalg.norm(v) > 1e-12:
            edge_vectors.append(v)
    return edge_vectors

def convex_hulls_overlap_3d(verticesA, verticesB):
    hullA = ConvexHull(verticesA)
    hullB = ConvexHull(verticesB)
    #
    normalsA = get_face_normals(hullA)
    normalsB = get_face_normals(hullB)
    #
    edgesA = get_edges(hullA)
    edgesB = get_edges(hullB)
    #
    axes = []
    #
    # 1. Face normals
    axes.extend(normalsA)
    axes.extend(normalsB)
    #
    # 2. Cross products of edges
    for e1 in edgesA:
        for e2 in edgesB:
            axis = np.cross(e1, e2)
            axis = normalize(axis)
            if axis is not None:
                axes.append(axis)
    #
    # SAT (Separating Axis Theorem) test: check all axes
    # SAT states that two convex shapes do NOT overlap if and only if there exists at least one axis on which their projections do not overlap.
    for axis in axes:
        amin, amax = project_points(verticesA, axis)
        bmin, bmax = project_points(verticesB, axis)
        #
        if not intervals_overlap(amin, amax, bmin, bmax):
            # Found separating axis → no overlap
            return False
    # No separating axis → they overlap
    return True

overlap_bool = convex_hulls_overlap_3d(hullv_pos_prog, hullv_pos_sec)
print('Overlapped:', overlap_bool)

#### Find a timestep that the two halos overlap (overlap_bool == True), but they don't at the preceeding timestep (overlap_bool == False) 

In [ ]:
# Visualize the results (boundary of the convex hulls in 2D projection)

hullA = ConvexHull(hullv_pos_prog)
hullB = ConvexHull(hullv_pos_sec)

# 2D plot to show the ConvexHull (y-z plane)
axis1 = 1
axis2 = 2

hullv_pos_vert  = np.append(ConvexHull(hullv_pos_prog[:,[axis1,axis2]]).vertices, ConvexHull(hullv_pos_prog[:,[axis1,axis2]]).vertices[0])
plt.plot(hullv_pos_prog[hullv_pos_vert][:,axis1], hullv_pos_prog[hullv_pos_vert][:,axis2], color='blue', label='ConvexHull created from hullv')
hullv_pos_vert  = np.append(ConvexHull(hullv_pos_sec[:,[axis1,axis2]]).vertices, ConvexHull(hullv_pos_sec[:,[axis1,axis2]]).vertices[0])
plt.plot(hullv_pos_sec[hullv_pos_vert][:,axis1], hullv_pos_sec[hullv_pos_vert][:,axis2], color='orange', label='ConvexHull created from hullv')
plt.show()